# Local DBpedia 2016-04 endpoint + execution evaluation

Runs entirely on this machine: downloads the DBpedia 2016-04 core dumps (the release LC-QuAD v1.0 was built against), builds a local Jena TDB2 triple store, serves it with Fuseki on `localhost:3030`, then executes every generated and gold query from `output.json`.

**What it produces:**
- `output.json` (project folder) gains `generated_results` and `gold_results` per record — SELECT → sorted URI list, ASK → bool, COUNT → int, `null` on failure
- `analytics.txt` (project folder) with exact answer match, per-type accuracy, macro (per-question) precision/recall/F1 over answer sets

**Prerequisites:** Java (installed ✓). `output.json` in the project folder — either from a local `test.py` run or downloaded from Drive after the Colab generation run. ~40 GB free disk during the build (store ends up ~20–30 GB in `dbpedia_endpoint/`).

**Timing:** downloads 1.2 GB (a few minutes on broadband), index build is the long step — roughly 1–2 h depending on disk speed, execution ~15 min. Unlike Colab there is no session limit, and the built store persists: to restart the endpoint later, just re-run the Fuseki cell.

**Data:** six 2016-04 files — instance types, mappingbased objects+literals (`ontology/` predicates), raw + mapped infobox properties (`property/` namespace), specific mappingbased properties. No wikilinks/labels/redirects (the predicate whitelist touches none of them); `instance_types_transitive` deliberately excluded so type facts match the original endpoint.

Notes: stored result lists capped at 2000 (metrics use the full sets); both-empty counts as a match (F1 = 1); failed/ERROR queries count as wrong. Run this notebook from the project root (VS Code or `jupyter notebook`).

In [90]:
from pathlib import Path
import shutil, subprocess
BEAMS = 1

# Two roots, because they move independently. A run's own files -- the queries
# test.py produced and the analytics written back -- sit beside this notebook,
# wherever it has been filed. The things it reads but never writes -- the T-box,
# the predicate whitelist, the Fuseki install and its TDB2 store -- live in the
# repo, which is whichever ancestor directory holds tbox_reasoner/. Finding the
# repo by that marker rather than assuming the notebook sits in its root means
# this works from the root and from a results folder alike.
HERE = Path.cwd()
PROJECT = next((p for p in (HERE, *HERE.parents) if (p / 'tbox_reasoner').is_dir()), None)
assert PROJECT is not None, (
    f'no tbox_reasoner/ in {HERE} or any parent -- start the kernel inside the repo')

WORK = PROJECT / 'dbpedia_endpoint'
DATA_DIR = WORK / 'data'          # downloaded + decompressed dumps
TDB_DIR = WORK / 'tdb2'           # the built triple store
ENDPOINT = 'http://localhost:3030/dbpedia/sparql'
OUT_JSON = HERE / f'{BEAMS}_output.json'
ANALYTICS = HERE / f'{BEAMS}_analytics.txt'

print(subprocess.run(['java', '-version'], capture_output=True, text=True).stderr.splitlines()[0])
print('repo:   ', PROJECT)
print('results:', HERE)
print('free disk:', shutil.disk_usage(PROJECT).free // 2**30, 'GB')
assert OUT_JSON.exists(), (
    f'{OUT_JSON.name} not found in {HERE} -- run test.py first, or download it from Drive')

openjdk version "25.0.2" 2026-01-20 LTS
free disk: 74 GB


In [91]:
# Reset <BEAMS>_output.json to generation output only, so everything below is
# recomputed from scratch. test.py writes the canonical queries and the
# string-match flags; the execution cell further down adds gold_results and
# <rung>_results. This strips those answer sets -- they are regenerable in a few
# minutes, and stale ones would silently mix a previous endpoint run into the new
# analytics. Idempotent: on an already-clean file it changes nothing.
import json

KEEP_EXACT = {'question', 'match_modulo_twins'}
KEEP_SUFFIX = ('_canonical', '_match')

records = json.loads(OUT_JSON.read_text(encoding='utf-8'))
before = sorted({k for r in records for k in r})
records = [{k: v for k, v in r.items() if k in KEEP_EXACT or k.endswith(KEEP_SUFFIX)}
           for r in records]
after = sorted({k for r in records for k in r})

dropped = [k for k in before if k not in after]
if dropped:
    size_before = OUT_JSON.stat().st_size
    OUT_JSON.write_text(json.dumps(records, indent=2), encoding='utf-8')
    print(f'{OUT_JSON.name}: removed {dropped}')
    print(f'  {size_before / 1e6:.1f} MB -> {OUT_JSON.stat().st_size / 1e6:.1f} MB')
else:
    print(f'{OUT_JSON.name}: already clean, nothing removed')

rungs = [k[:-len('_canonical')] for k in after
         if k.endswith('_canonical') and k != 'gold_canonical']
print(f'{len(records)} records | rungs present: {rungs}')
print(f'fields kept: {after}')

1_output.json: already clean, nothing removed
1000 records | rungs present: ['generated', 'grammar_only', 'no_boosts', 'unconstrained']
fields kept: ['generated_canonical', 'generated_match', 'gold_canonical', 'grammar_only_canonical', 'grammar_only_match', 'match_modulo_twins', 'no_boosts_canonical', 'no_boosts_match', 'question', 'unconstrained_canonical', 'unconstrained_match']


In [92]:
# download + extract Jena and Fuseki (pinned from the Apache archive)
import tarfile, urllib.request

JENA_VER = '5.5.0'
JENA = WORK / f'apache-jena-{JENA_VER}'
FUSEKI = WORK / f'apache-jena-fuseki-{JENA_VER}'

def download(url, dest):
    if dest.exists():
        print('already have', dest.name)
        return
    print('downloading', dest.name, flush=True)
    with urllib.request.urlopen(url) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f, 1024 * 1024)

WORK.mkdir(exist_ok=True)
for name in (f'apache-jena-{JENA_VER}', f'apache-jena-fuseki-{JENA_VER}'):
    tgz = WORK / f'{name}.tar.gz'
    download(f'https://archive.apache.org/dist/jena/binaries/{name}.tar.gz', tgz)
    if not (WORK / name).exists():
        with tarfile.open(tgz) as t:
            t.extractall(WORK, filter='data')
        print('extracted', name)
print('jena + fuseki ready')

already have apache-jena-5.5.0.tar.gz
already have apache-jena-fuseki-5.5.0.tar.gz
jena + fuseki ready


In [93]:
# download the six 2016-04 dump files (~1.2 GB total) and decompress (~17 GB)
FILES = [
    'instance_types_en.ttl.bz2',               # rdf:type triples
    'mappingbased_objects_en.ttl.bz2',         # ontology/ predicates -> entity objects
    'mappingbased_literals_en.ttl.bz2',        # ontology/ predicates -> literal values
    'infobox_properties_en.ttl.bz2',           # raw property/ namespace
    'infobox_properties_mapped_en.ttl.bz2',    # property/ with normalized values
    'specific_mappingbased_properties_en.ttl.bz2',
]
DATA_DIR.mkdir(parents=True, exist_ok=True)
for f in FILES:
    download(f'https://downloads.dbpedia.org/2016-04/core-i18n/en/{f}', DATA_DIR / f)

import bz2
for bz in sorted(DATA_DIR.glob('*.bz2')):
    out = bz.with_suffix('')  # strips .bz2 -> .ttl
    if out.exists():
        continue
    with bz2.open(bz, 'rb') as fin, open(out, 'wb') as fout:
        shutil.copyfileobj(fin, fout, 1024 * 1024)
    print('decompressed', out.name)

already have instance_types_en.ttl.bz2
already have mappingbased_objects_en.ttl.bz2
already have mappingbased_literals_en.ttl.bz2
already have infobox_properties_en.ttl.bz2
already have infobox_properties_mapped_en.ttl.bz2
already have specific_mappingbased_properties_en.ttl.bz2


In [7]:
# build the TDB2 index -- the long step (~1-2 h). Loader progress streams
# into this cell; riot WARN lines about odd IRIs/literals are harmless
# (messy raw infobox data, a handful of triples per million).
import os

os.environ['JVM_ARGS'] = '-Xmx8g'  # loader heap; raise to 12g if you have 32 GB RAM
ttls = [str(p) for p in sorted(DATA_DIR.glob('*.ttl'))]
assert len(ttls) == 6, f'expected 6 .ttl files, found {len(ttls)}'
cmd = [str(JENA / 'bat' / 'tdb2_tdbloader.bat'), '--loc', str(TDB_DIR)] + ttls
proc = subprocess.Popen(' '.join(f'"{c}"' for c in cmd), shell=True,
                        env={**os.environ, 'JENA_HOME': str(JENA)},
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0 or not TDB_DIR.exists():
    raise RuntimeError('tdbloader failed -- see output above; do NOT continue to the Fuseki cell')
print('store built at', TDB_DIR)

12:32:28 INFO  loader          :: Loader = LoaderPhased
12:32:28 INFO  loader          :: Start: 6 files
12:32:29 WARN  riot            :: [line: 19205, col: 76] Bad IRI: <http:/www.debian.org/> Code: 57/REQUIRED_COMPONENT_MISSING in HOST: A component that is required by the scheme is missing.
12:32:29 WARN  riot            :: [line: 30784, col: 74] Bad IRI: <http:/www.gnu.org/software/gzip/> Code: 57/REQUIRED_COMPONENT_MISSING in HOST: A component that is required by the scheme is missing.
12:32:29 WARN  riot            :: [line: 47011, col: 75] Bad IRI: <http:/latex-project.org/> Code: 57/REQUIRED_COMPONENT_MISSING in HOST: A component that is required by the scheme is missing.
12:32:29 WARN  riot            :: [line: 61272, col: 73] Bad IRI: <http:/php.net> Code: 57/REQUIRED_COMPONENT_MISSING in HOST: A component that is required by the scheme is missing.
12:32:29 WARN  riot            :: [line: 66747, col: 78] Bad IRI: <http:/www.redhat.com> Code: 57/REQUIRED_COMPONENT_MISSING in H

In [94]:
# start Fuseki on localhost and wait for it; then a sanity count
import json, time, urllib.parse, urllib.request

fuseki_log = open(WORK / 'fuseki.log', 'w')
fuseki = subprocess.Popen(f'"{FUSEKI / "fuseki-server.bat"}" --tdb2 --loc "{TDB_DIR}" /dbpedia',
                          shell=True, cwd=FUSEKI, stdout=fuseki_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:3030/$/ping', timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Fuseki did not start -- check dbpedia_endpoint/fuseki.log')

def sparql(q, timeout=300):
    url = ENDPOINT + '?query=' + urllib.parse.quote(q)
    req = urllib.request.Request(url, headers={'Accept': 'application/sparql-results+json'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

resp = sparql('SELECT (COUNT(*) AS ?c) WHERE { ?s <http://dbpedia.org/ontology/architect> ?o }')
print('dbo:architect triples in store:', resp['results']['bindings'][0]['c']['value'])
print('endpoint up at', ENDPOINT)

dbo:architect triples in store: 17237
endpoint up at http://localhost:3030/dbpedia/sparql


In [95]:
# execute gold and all four generation modes for every record, enrich
# <BEAMS>_output.json with their answer sets, write <BEAMS>_analytics.txt
STORE_CAP = 2000  # metrics below always use the FULL sets; only the stored record is capped

records = json.loads(OUT_JSON.read_text(encoding='utf-8'))

import re

def execute(q):
    """SELECT -> sorted list of answer values; ASK -> bool; COUNT -> int; failure -> None."""
    if q.startswith('ERROR:'):  # generation itself failed on this question
        return None
    is_count = 'COUNT(' in q.upper()
    if is_count:
        # Jena quirk: 'SELECT DISTINCT COUNT(...)' without an alias returns an
        # EMPTY binding (DISTINCT over the unaliased aggregate drops the value).
        # Alias it explicitly so the count comes back; applied to generated and
        # gold alike, so the comparison stays fair.
        q = re.sub(r'SELECT\s+DISTINCT\s+COUNT\(([^)]*)\)',
                   lambda m: f'SELECT (COUNT({m.group(1).strip()}) AS ?__count__)',
                   q, flags=re.IGNORECASE)
    try:
        resp = sparql(q)
    except Exception:
        return None
    if 'boolean' in resp:
        return bool(resp['boolean'])
    bindings, var = resp['results']['bindings'], resp['head']['vars'][0]
    if is_count:
        return int(bindings[0][var]['value']) if bindings and bindings[0] else 0
    return sorted(b[var]['value'] for b in bindings if var in b)

def kind(q):
    u = q.lstrip().upper()
    return 'ask' if u.startswith('ASK') else ('count' if 'COUNT(' in u else 'select')

# every rung of the ablation ladder, strongest first. test.py writes one
# <name>_canonical per system plus <name>_match (canonical string equality
# against gold), so execution here only has to add the answer sets
SYSTEMS = (
    ("generated", "CONSTRAINED (grammar + tries + ontological boosts):"),
    ("no_boosts", "NO BOOSTS (grammar + tries, no domain/range guidance):"),
    ("grammar_only", "GRAMMAR ONLY (query structure alone, free-form IRIs):"),
    ("unconstrained", "UNCONSTRAINED (raw fine-tuned model, no constraints):"),
)
missing = [n for n, _ in SYSTEMS if f"{n}_canonical" not in records[0]]
assert not missing, f"{OUT_JSON.name} has no {missing} -- rerun test.py"

gold_all = []
full = {name: [] for name, _ in SYSTEMS}
t0 = time.time()
for i, rec in enumerate(records, 1):
    gold = execute(rec["gold_canonical"])
    gold_all.append(gold)
    rec["gold_results"] = gold[:STORE_CAP] if isinstance(gold, list) else gold
    for name, _ in SYSTEMS:
        produced = execute(rec[f"{name}_canonical"])
        full[name].append(produced)
        rec[f"{name}_results"] = produced[:STORE_CAP] if isinstance(produced, list) else produced
    if i % 50 == 0:
        OUT_JSON.write_text(json.dumps(records, indent=2), encoding="utf-8")
        print(f"{i}/{len(records)} executed ({(time.time() - t0) / i:.2f}s per record, "
              f"{len(SYSTEMS) + 1} queries each)", flush=True)
OUT_JSON.write_text(json.dumps(records, indent=2), encoding="utf-8")

# ---- metrics ----
n = len(records)
types = {k: sum(kind(r["gold_canonical"]) == k for r in records) for k in ("select", "count", "ask")}
gold_failed = sum(gold is None for gold in gold_all)


def summarize(name, label, idx=None):
    """The metric block for one rung against gold, as report lines.

    kind() always keys off the gold query, so the per-type totals are identical
    across rungs -- which is what makes the blocks comparable line for line.

    idx restricts the block to a subset of question indices; the default is
    every question. The ontological-compatibility cell below passes subsets in,
    so that split is scored by this function and not by a second copy of the
    metric code."""
    rows = range(len(records)) if idx is None else list(idx)
    m = len(rows)
    stats = {"ask": [0, 0], "count": [0, 0], "select": [0, 0]}  # [correct, total]
    qmatch = {"ask": 0, "count": 0, "select": 0}
    exact_overall = query_overall = both_empty = failed = 0
    # Macro averaging: P/R/F1 per question, then averaged, so every question
    # counts once however large its answer set. The QALD/GERBIL convention the
    # LC-QuAD literature reports against.
    sp = sr = sf = 0.0
    qald = 0.0  # per-question F1 over ALL questions, ASK/COUNT scored 0/1

    for i in rows:
        rec, produced, gold = records[i], full[name][i], gold_all[i]
        k = kind(rec["gold_canonical"])
        stats[k][1] += 1
        if produced is None:
            failed += 1
        if produced is not None and produced == gold:
            exact_overall += 1
            stats[k][0] += 1
        # query-level match was already decided by test.py: canonical string
        # equality. Strictly stronger than answer equality (a wrong query can
        # still return the right answer) and blind to equivalent queries spelled
        # differently, so read it as a conservative lower bound
        if rec[f"{name}_match"]:
            query_overall += 1
            qmatch[k] += 1
        if k == "select":
            g = set(gold) if isinstance(gold, list) else set()
            p = set(produced) if isinstance(produced, list) else set()
            tp = len(g & p)
            if not g and not p:
                both_empty += 1
            # Empty-answer convention follows GERBIL/QALD, matching the LC-QuAD
            # systems we compare against: empty prediction -> P=1 (F1 still 0 via
            # R=0); empty gold -> R=1. This is why macro precision sits far above
            # recall -- convention, not bug.
            pr = tp / len(p) if p else 1.0
            rc = tp / len(g) if g else 1.0
            f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
            sp += pr
            sr += rc
            sf += f1
            qald += f1
        else:
            # ASK is a single boolean and COUNT a single integer: no partial
            # overlap to score, so per-question F1 is 1 when the answer matches
            # and 0 otherwise -- how QALD folds them in with the SELECTs
            qald += 1.0 if (produced is not None and produced == gold) else 0.0

    sel_n = stats["select"][1]

    def per_type(correct):
        rows = []
        for k in ("select", "count", "ask"):
            c, t = correct[k], stats[k][1]
            rows.append(f"    {k:6} {c}/{t} ({c / t:.1%})" if t else f"    {k:6} 0/0")
        return rows

    out = [
        label,
        f"  execution failures: {failed}",
        f"  QALD macro F1 over all {m} questions (ASK/COUNT scored 0/1): {qald / m:.3f}",
        "",
        f"  exact answer match: {exact_overall}/{m} ({exact_overall / m:.1%})",
    ]
    out += per_type({k: v[0] for k, v in stats.items()})
    out += [f"  exact query match, canonicalised: {query_overall}/{m} ({query_overall / m:.1%})"]
    out += per_type(qmatch)
    out += [
        f"  SELECT answer sets (n={sel_n}), macro-averaged over questions:",
        f"    precision {sp / sel_n:.3f}  recall {sr / sel_n:.3f}  F1 {sf / sel_n:.3f}"
        if sel_n else "    n/a",
        f"    both empty (produced & gold): {both_empty}",
    ]
    return out


lines = [
    "LC-QuAD v1.0 test -- execution evaluation",
    "endpoint: local Jena Fuseki, DBpedia 2016-04 core (en)  " + time.strftime("%Y-%m-%d %H:%M"),
    f"beam width: {BEAMS}",
    "",
    f"questions: {n}  (select: {types['select']}, count: {types['count']}, ask: {types['ask']})",
    f"gold execution failures: {gold_failed}",
    "",
    "The ablation ladder: generated - no_boosts isolates the ontological boosts,",
    "no_boosts - grammar_only isolates the KB vocabulary (entity tries + relation",
    "whitelist), grammar_only - unconstrained isolates query structure alone.",
    "",
]
for name, label in SYSTEMS:
    lines += summarize(name, label) + [""]

# headline table, so the rungs can be read side by side without scrolling
lines += ["SUMMARY (QALD macro F1 | exact answer | exact query):"]
for name, label in SYSTEMS:
    block = summarize(name, label)
    qald_line = next(l for l in block if "QALD macro F1" in l).split(": ")[1]
    ans = next(l for l in block if "exact answer match" in l).split(": ")[1]
    qry = next(l for l in block if "exact query match" in l).split(": ")[1]
    lines.append(f"  {name:14} {qald_line:>6}  |  {ans:>18}  |  {qry:>18}")

text = "\n".join(lines)
print(text)
ANALYTICS.write_text(text, encoding="utf-8")
print("written to", ANALYTICS)


50/1000 executed (0.15s per record, 5 queries each)
100/1000 executed (0.13s per record, 5 queries each)
150/1000 executed (0.12s per record, 5 queries each)
200/1000 executed (0.12s per record, 5 queries each)
250/1000 executed (0.11s per record, 5 queries each)
300/1000 executed (0.13s per record, 5 queries each)
350/1000 executed (0.13s per record, 5 queries each)
400/1000 executed (0.13s per record, 5 queries each)
450/1000 executed (0.12s per record, 5 queries each)
500/1000 executed (0.12s per record, 5 queries each)
550/1000 executed (0.12s per record, 5 queries each)
600/1000 executed (0.12s per record, 5 queries each)
650/1000 executed (0.12s per record, 5 queries each)
700/1000 executed (0.13s per record, 5 queries each)
750/1000 executed (0.13s per record, 5 queries each)
800/1000 executed (0.13s per record, 5 queries each)
850/1000 executed (0.13s per record, 5 queries each)
900/1000 executed (0.13s per record, 5 queries each)
950/1000 executed (0.13s per record, 5 queries 

In [96]:
# string-match metrics over output.json: strict exact match and match modulo
# namespace twins. Code copied from test.py (test.py cannot be imported here --
# it imports the model-loading generation module); predicates.txt path adjusted
import json
import re


def canonicalize(q):
    # _scratch_trie.canonicalize plus one extra step: drop a trailing dot
    # before the closing brace. Legal SPARQL and present in ~28% of gold
    # queries, but the generator's grammar can never emit it, so without
    # this those queries could never exact-match.
    q = " ".join(q.split())
    q = q.replace("COUNT( ?uri )", "COUNT(?uri)")
    q = q.replace("{", "{ ").replace("}", " }")
    q = re.sub(r"\s*\.\s*(?![^<]*>)", " . ", q)
    return " ".join(q.split()).replace(" . }", " }")


def _load_twin_names():
    # predicate local-names whitelisted in BOTH the ontology/ and property/
    # namespaces -- the pairs a strict exact match cannot tell apart
    ont, prop = set(), set()
    for line in (PROJECT / "lcquad_data" / "predicates.txt").read_text(encoding="utf-8").splitlines():
        iri = line.strip().rstrip(",")
        if "/ontology/" in iri:
            ont.add(iri.rsplit("/", 1)[1])
        elif "/property/" in iri:
            prop.add(iri.rsplit("/", 1)[1])
    return ont & prop


TWIN_RE = re.compile(
    r"<http://dbpedia\.org/(?:ontology|property)/("
    + "|".join(sorted((re.escape(n) for n in _load_twin_names()), key=len, reverse=True))
    + r")>"
)


def canonicalize_twins(q):
    # canonicalize, then rewrite every twin predicate to a namespace-neutral
    # IRI, so gold <.../property/architect> and generated <.../ontology/architect>
    # compare equal. Generation itself is unaffected.
    return TWIN_RE.sub(r"<dbpedia-twin/\1>", canonicalize(q))


records = json.loads(OUT_JSON.read_text(encoding="utf-8"))


def string_metrics(name, label):
    """exact / twin-neutral string match against gold for one rung. The exact
    figure is the one test.py already decided; only the twin-neutral variant
    needs recomputing, since the stored queries are canonical already."""
    n = len(records)
    exact = sum(r[f"{name}_match"] for r in records)
    twins = sum(canonicalize_twins(r[f"{name}_canonical"])
                == canonicalize_twins(r["gold_canonical"]) for r in records)
    print(label)
    print(f"  exact match:        {exact}/{n} ({exact / n:.1%})")
    print(f"  match modulo twins: {twins}/{n} ({twins / n:.1%})")


for name, label in SYSTEMS:   # SYSTEMS comes from the execution cell above
    string_metrics(name, label)
    print()


CONSTRAINED (grammar + tries + ontological boosts):
  exact match:        256/1000 (25.6%)
  match modulo twins: 364/1000 (36.4%)

NO BOOSTS (grammar + tries, no domain/range guidance):
  exact match:        257/1000 (25.7%)
  match modulo twins: 364/1000 (36.4%)

GRAMMAR ONLY (query structure alone, free-form IRIs):
  exact match:        247/1000 (24.7%)
  match modulo twins: 351/1000 (35.1%)

UNCONSTRAINED (raw fine-tuned model, no constraints):
  exact match:        247/1000 (24.7%)
  match modulo twins: 351/1000 (35.1%)



In [97]:
# T-box guidance where the ontology actually has something to say.
#
# The headline boost effect (generated - no_boosts) is ~0, but 419 of 616
# relations carry owl:Thing-only domains, so on most questions the T-box has
# nothing to work with and the average is diluted. This splits the test set by
# whether the GOLD predicates are informative -- informative meaning the
# effective domain or range names at least one class other than owl:Thing --
# and reports every rung separately on each part. If the boost fails only
# because of vacuity, it should work on the informative part; the "none
# informative" row is the control where it cannot possibly work.
#
# Namespace-neutral match reuses canonicalize_twins() from the cell above, so
# these figures are directly comparable with the mod-twins numbers there. That
# rule only neutralises local names whitelisted in BOTH namespaces, which is
# deliberately conservative -- it never merges two predicates that are not
# genuinely a twin pair.
#
# The block is appended to <BEAMS>_analytics.txt after the metrics written by
# the execution cell. Re-running replaces the previous copy rather than
# stacking another one, so the cell is safe in a Run-all.
import json

_tbox = json.loads((PROJECT / 'tbox_reasoner' / 'tbox_rules.json').read_text(encoding='utf-8'))
_DOMAIN = _tbox['effective_property_domain_map']
_RANGE = _tbox['effective_property_range_map']
OWL_THING = '<http://www.w3.org/2002/07/owl#Thing>'
RDF_TYPE = '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>'


def query_predicates(q):
    """Predicates of a canonical query, rdf:type tails excluded -- rdf:type has
    no domain or range of its own in the map, and counting it would misclassify
    every query that carries a type tail."""
    i = q.find('WHERE { ')
    if i < 0:
        return []
    body = q[i + len('WHERE { '):].rstrip()
    if body.endswith('}'):
        body = body[:-1].rstrip()
    out = []
    for part in body.split(' . '):
        t = part.split()
        if len(t) == 3 and t[1] != RDF_TYPE:
            out.append(t[1])
    return out


def informative(p):
    """True when the ontology says something usable about this predicate."""
    return (any(c != OWL_THING for c in (_DOMAIN.get(p) or []))
            or any(c != OWL_THING for c in (_RANGE.get(p) or [])))


def subset(rule):
    """Question indices whose gold predicates satisfy rule() over their
    informativeness. Queries with no predicate at all are excluded."""
    out = []
    for i, rec in enumerate(records):
        preds = query_predicates(rec['gold_canonical'])
        if preds and rule(informative(p) for p in preds):
            out.append(i)
    return out


SUBSETS = [
    ('every gold predicate informative', subset(all)),
    ('at least one informative', subset(any)),
    ('none informative (control)', subset(lambda flags: not any(flags))),
]

have_answers = 'gold_results' in records[0]
lines = [
    'T-box guidance on questions where the ontology has something to say',
    '',
    'Subset defined by the GOLD predicates; a predicate counts as informative when',
    'its effective domain or range names at least one class other than owl:Thing.',
    'rdf:type tails are excluded -- they carry no domain or range of their own.',
    'Namespace-neutral match uses the same conservative twin rule as the block',
    'above (only local names whitelisted in both namespaces are collapsed).',
    '',
    'The bottom subset is the control: the T-box cannot help there, so a boost',
    'delta of the same size on both is evidence that vacuity is not the binding',
    'constraint. Note the subset is defined by gold predicates while the model may',
    'emit others -- a proxy, not a controlled experiment.',
    '',
]

for title, idx in SUBSETS:
    lines.append(f'{title}: n={len(idx)} ({len(idx) / len(records):.1%} of {len(records)})')
    if not idx:
        lines += ['  empty', '']
        continue
    header = f"  {'rung':14} {'exact query':>16} {'modulo namespace':>18}"
    if have_answers:
        header += f" {'exact answer':>16}"
    lines.append(header)
    scores = {}
    for rung, _label in SYSTEMS:
        ex = sum(records[i][f'{rung}_match'] for i in idx)
        ns = sum(canonicalize_twins(records[i][f'{rung}_canonical'])
                 == canonicalize_twins(records[i]['gold_canonical']) for i in idx)
        row = f'  {rung:14} {ex:8} ({ex / len(idx):5.1%}) {ns:10} ({ns / len(idx):5.1%})'
        an = None
        if have_answers:
            an = sum(records[i][f'{rung}_results'] == records[i]['gold_results'] for i in idx)
            row += f' {an:8} ({an / len(idx):5.1%})'
        scores[rung] = (ex, ns, an)
        lines.append(row)
    if 'generated' in scores and 'no_boosts' in scores:
        b, nb = scores['generated'], scores['no_boosts']
        delta = f'exact {b[0] - nb[0]:+d}, modulo-namespace {b[1] - nb[1]:+d}'
        if have_answers:
            delta += f', answer {b[2] - nb[2]:+d}'
        lines.append(f'  T-box guidance (generated - no_boosts): {delta}')
    lines.append('')


text = '\n'.join(lines)
print(text)

# append to the analytics file, replacing any previous copy of this block so a
# re-run does not stack duplicates
MARKER = lines[0]
prev = ANALYTICS.read_text(encoding='utf-8') if ANALYTICS.exists() else ''
cut = prev.find(MARKER)
if cut != -1:
    prev = prev[:cut]
prev = prev.rstrip('\n')
ANALYTICS.write_text((prev + '\n\n' if prev else '') + text + '\n', encoding='utf-8')
print('\nappended to', ANALYTICS)


T-box guidance on questions where the ontology has something to say

Subset defined by the GOLD predicates; a predicate counts as informative when
its effective domain or range names at least one class other than owl:Thing.
rdf:type tails are excluded -- they carry no domain or range of their own.
Namespace-neutral match uses the same conservative twin rule as the block
above (only local names whitelisted in both namespaces are collapsed).

The bottom subset is the control: the T-box cannot help there, so a boost
delta of the same size on both is evidence that vacuity is not the binding
constraint. Note the subset is defined by gold predicates while the model may
emit others -- a proxy, not a controlled experiment.

every gold predicate informative: n=342 (34.2% of 1000)
  rung                exact query   modulo namespace     exact answer
  generated           105 (30.7%)        147 (43.0%)      154 (45.0%)
  no_boosts           106 (31.0%)        146 (42.7%)      153 (44.7%)
  gramma

In [98]:
# Ontological compatibility of the GOLD query -- and the whole metric ladder
# on each half.
#
# The boosts charge a hypothesis twice over: RELATION_BOOST when a relation's
# effective domain is not covered by the subject's types, OBJECT_BOOST when the
# object falls outside the relation's effective range. LC-QuAD was not authored
# against the DBpedia ontology, so the gold query itself sometimes breaks those
# rules -- and where it does, a decoder that prices ontological compliance is
# being steered away from the target. Splitting the test set by whether GOLD
# would be charged therefore separates "the guidance is wrong" from "the
# benchmark is", which the informative/uninformative split above cannot: the
# control subset there is exactly the subset where the T-box CANNOT contradict
# gold, so it collects credit for suppressing wrong alternatives and can never
# be charged itself.
#
# Both checks mirror type_constrained_generation.py:
#   domain  entity subjects only -- a variable subject sets prev to ?uri/?x,
#           which switches the relation boost off, so nothing is charged
#   range   entity objects only -- a variable object is exempt, and an
#           owl:Thing range produces no range tries and so no charge
#   a type covers a domain when it IS that class or a descendant of it; a range
#           admits its own class plus every subclass, because the class tries
#           are partitioned by most-specific direct type and a superclass trie
#           does not contain its subclasses
#   rdf:type tails are skipped: the type tail is its own grammar branch, not a
#           whitelisted relation, and carries no domain or range
#
# Types come from the endpoint, cached in <BEAMS>_entity_types.json -- the same
# cache tbox_violations.ipynb writes, so the two notebooks share it. It holds
# the most-specific ontology type, which is what the decoder's class tries hold.
# A gold predicate outside the whitelist has no domain or range in the map and
# so is never charged; the decoder could not have emitted it either way.
import json

assert 'full' in globals() and 'gold_all' in globals(), \
    'run the execution cell above first -- this cell reuses its uncapped answer sets'

_ANCESTORS = {}
for _parent, _descendants in _tbox['class_subsumptions'].items():
    for _child in _descendants:
        _ANCESTORS.setdefault(_child, set()).add(_parent)
_SUBCLASSES = _tbox['class_subsumptions']
_TBOX_CLASSES = set(_tbox['classes'])
TYPE_CACHE = PROJECT / f'{BEAMS}_entity_types.json'


def query_triples(q):
    """(subject, predicate, object) per pattern. Canonicalisation guarantees
    ' . ' between patterns and single spaces inside them, so no parser needed."""
    i = q.find('WHERE { ')
    if i < 0:
        return []
    body = q[i + len('WHERE { '):].rstrip()
    if body.endswith('}'):
        body = body[:-1].rstrip()
    out = []
    for part in body.split(' . '):
        t = part.split()
        if len(t) == 3:
            out.append(tuple(t))
    return out


def is_entity(term):
    return term.startswith('<') and '/resource/' in term


# --- resolve the type of every entity that appears in a GOLD query ---
_wanted = {t for rec in records for tr in query_triples(rec['gold_canonical'])
           for t in (tr[0], tr[2]) if is_entity(t)}
entity_types = json.loads(TYPE_CACHE.read_text(encoding='utf-8')) if TYPE_CACHE.exists() else {}
_todo = sorted(_wanted - entity_types.keys())
print(f'{len(_wanted)} distinct entities in gold queries; {len(_todo)} to look up')

_BATCH = 200
for _i in range(0, len(_todo), _BATCH):
    _chunk = _todo[_i:_i + _BATCH]
    _res = sparql('SELECT ?e ?t WHERE { VALUES ?e { ' + ' '.join(_chunk) + ' } ?e a ?t }')
    for _e in _chunk:
        entity_types.setdefault(_e, [])      # looked up; empty means genuinely untyped
    for _b in _res['results']['bindings']:
        _e, _t = '<' + _b['e']['value'] + '>', '<' + _b['t']['value'] + '>'
        _got = entity_types.setdefault(_e, [])   # setdefault again: an IRI can echo
        if _t in _TBOX_CLASSES and _t not in _got:   # back in a form the chunk did not use
            _got.append(_t)
if _todo:
    TYPE_CACHE.write_text(json.dumps(entity_types), encoding='utf-8')
_untyped = sum(not entity_types[e] for e in _wanted)
print(f'{_untyped}/{len(_wanted)} of them carry no DBpedia ontology type '
      f'({_untyped / len(_wanted):.1%}) -- those fail the domain check, as they do in the decoder')


def domain_covered(pred, subject):
    """RELATION_BOOST's test, from encouraged_relations()."""
    domains = [c for c in (_DOMAIN.get(pred) or []) if c != OWL_THING]
    if not domains or not is_entity(subject):
        return True
    types = entity_types.get(subject, [])
    return all(any(d == t or d in _ANCESTORS.get(t, ()) for t in types) for d in domains)


def range_admits(pred, obj):
    """OBJECT_BOOST's test, from range_tries()."""
    wanted = [c for c in (_RANGE.get(pred) or []) if c != OWL_THING]
    if not wanted or not is_entity(obj):
        return True
    allowed = set()
    for c in wanted:
        allowed.add(c)
        allowed.update(_SUBCLASSES.get(c, []))
    return any(t in allowed for t in entity_types.get(obj, []))


def gold_charge(rec):
    """Which of the two boosts would charge this gold query, if either."""
    why = set()
    for s, p, o in query_triples(rec['gold_canonical']):
        if p == RDF_TYPE:
            continue
        if not domain_covered(p, s):
            why.add('domain')
        if not range_admits(p, o):
            why.add('range')
    return why


_charge = [gold_charge(rec) for rec in records]
COMPATIBLE = [i for i, w in enumerate(_charge) if not w]
INCOMPATIBLE = [i for i, w in enumerate(_charge) if w]
_n = len(records)

lines = [
    'Ontological compatibility of the gold query',
    '',
    'A gold query is INCOMPATIBLE when the decoder\'s own boosts would charge it:',
    'a relation whose effective domain the subject\'s types do not cover, or an',
    'object outside the relation\'s effective range. On those questions the boosts',
    'push away from gold by construction, so the two halves answer different',
    'questions -- does the guidance help when the benchmark agrees with the',
    'ontology, and how much does it cost when the benchmark does not.',
    '',
    'Types are endpoint-resolved most-specific ontology types, the same ones the',
    'decoder\'s class tries hold. Untyped entities fail the domain check here',
    'exactly as they do in the decoder.',
    '',
    f'gold compatible:   {len(COMPATIBLE)}/{_n} ({len(COMPATIBLE) / _n:.1%})',
    f'gold incompatible: {len(INCOMPATIBLE)}/{_n} ({len(INCOMPATIBLE) / _n:.1%})',
]
for _why, _label in ((('domain',), 'domain only'), (('range',), 'range only'),
                     (('domain', 'range'), 'both')):
    _k = sum(1 for w in _charge if w == set(_why))
    lines.append(f'    charged on {_label:12} {_k:4d}')
lines.append('')

for _title, _idx in (('GOLD ONTOLOGICALLY COMPATIBLE', COMPATIBLE),
                     ('GOLD ONTOLOGICALLY INCOMPATIBLE', INCOMPATIBLE)):
    lines += ['=' * 74,
              f'{_title}: n={len(_idx)} ({len(_idx) / _n:.1%} of {_n})',
              '=' * 74, '']
    if not _idx:
        lines += ['  empty', '']
        continue
    for _name, _label in SYSTEMS:
        lines += summarize(_name, _label, _idx) + ['']
    lines.append('SUMMARY on this subset (QALD macro F1 | exact answer | exact query):')
    for _name, _label in SYSTEMS:
        _block = summarize(_name, _label, _idx)
        _q = next(l for l in _block if 'QALD macro F1' in l).split(': ')[1]
        _a = next(l for l in _block if 'exact answer match' in l).split(': ')[1]
        _e = next(l for l in _block if 'exact query match' in l).split(': ')[1]
        lines.append(f'  {_name:14} {_q:>6}  |  {_a:>18}  |  {_e:>18}')
    # the boost delta, counted the same way summarize() counts it
    _ans = {r: sum(full[r][i] is not None and full[r][i] == gold_all[i] for i in _idx)
            for r, _ in SYSTEMS}
    _qry = {r: sum(records[i][f'{r}_match'] for i in _idx) for r, _ in SYSTEMS}
    lines += ['',
              f'  T-box guidance (generated - no_boosts): '
              f'answer {_ans["generated"] - _ans["no_boosts"]:+d}, '
              f'query {_qry["generated"] - _qry["no_boosts"]:+d}',
              '']

text = '\n'.join(lines)
print(text)

# append to the analytics file, replacing any previous copy of this block so a
# re-run does not stack duplicates
MARKER = lines[0]
prev = ANALYTICS.read_text(encoding='utf-8') if ANALYTICS.exists() else ''
cut = prev.find(MARKER)
if cut != -1:
    prev = prev[:cut]
prev = prev.rstrip('\n')
ANALYTICS.write_text((prev + '\n\n' if prev else '') + text + '\n', encoding='utf-8')
print('\nappended to', ANALYTICS)


1165 distinct entities in gold queries; 1165 to look up
82/1165 of them carry no DBpedia ontology type (7.0%) -- those fail the domain check, as they do in the decoder
Ontological compatibility of the gold query

A gold query is INCOMPATIBLE when the decoder's own boosts would charge it:
a relation whose effective domain the subject's types do not cover, or an
object outside the relation's effective range. On those questions the boosts
push away from gold by construction, so the two halves answer different
questions -- does the guidance help when the benchmark agrees with the
ontology, and how much does it cost when the benchmark does not.

Types are endpoint-resolved most-specific ontology types, the same ones the
decoder's class tries hold. Untyped entities fail the domain check here
exactly as they do in the decoder.

gold compatible:   885/1000 (88.5%)
gold incompatible: 115/1000 (11.5%)
    charged on domain only    27
    charged on range only     88
    charged on both          

In [ ]:
# deterministic analogue of the four accuracy axes the TEXT2SPARQL 26
# researcher-agents paper (arXiv:2608.07700) scores with an LLM judge:
# BGP nodes (entity/class IRIs), BGP predicates (twin-neutral), inner
# operators (FILTER/OPTIONAL/UNION/MINUS), outer operators (query form,
# COUNT, ORDER BY/LIMIT). Their number is LLM-judged; this table is
# string/structure-based, so the two are comparable in spirit only.
# Uses canonicalize_twins and records from the cells above.

def parse(q):
    """twin-neutral canonical form -> (head, [(s, p, o), ...]), or None if
    the query does not fit the five-template shape (e.g. ERROR)"""
    qc = canonicalize_twins(q)
    if "WHERE { " not in qc or not qc.endswith(" }"):
        return None
    head, _, body = qc.partition("WHERE { ")
    triples, cur = [], []
    for t in body[:-2].split():
        if t == ".":
            triples.append(tuple(cur))
            cur = []
        else:
            cur.append(t)
    triples.append(tuple(cur))
    if any(len(tr) != 3 for tr in triples):
        return None
    return head.strip(), triples


INNER_OPS = ("FILTER", "OPTIONAL", "UNION", "MINUS")
OUTER_OPS = ("ORDER BY", "LIMIT", "OFFSET")


def axes(gen, gold):
    """per-axis correctness booleans for one (generated, gold) query pair;
    an unparseable query is wrong on every axis"""
    g, d = parse(gen), parse(gold)
    if g is None or d is None:
        return {"nodes": False, "preds": False, "inner": False, "outer": False}
    nodes = lambda trs: sorted({t for tr in trs for t in (tr[0], tr[2]) if t.startswith("<")})
    preds = lambda trs: sorted(tr[1] for tr in trs)
    ops = lambda q, kws: {kw for kw in kws if kw in q.upper()}
    return {
        "nodes": nodes(g[1]) == nodes(d[1]),
        "preds": preds(g[1]) == preds(d[1]),
        "inner": ops(gen, INNER_OPS) == ops(gold, INNER_OPS),
        "outer": (g[0], ops(gen, OUTER_OPS)) == (d[0], ops(gold, OUTER_OPS)),
    }


def axes_table(name, label):
    """the four axes plus all-four-correct, for one system's queries"""
    counts = {"nodes": 0, "preds": 0, "inner": 0, "outer": 0, "overall": 0}
    for r in records:
        a = axes(r[f"{name}_canonical"], r["gold_canonical"])
        for k, v in a.items():
            counts[k] += v
        counts["overall"] += all(a.values())
    n = len(records)
    print(label)
    for k, c in counts.items():
        print(f"  {k:<8} {c / n:.1%}  ({c}/{n})")


for name, label in SYSTEMS:   # SYSTEMS comes from the execution cell above
    axes_table(name, label)
    print()


**Afterwards:** the Fuseki server keeps running in the background while this kernel is alive — interrupt the kernel (or close Jupyter) to stop it. The store in `dbpedia_endpoint/tdb2` persists on disk, so next time you only need to re-run the setup cell and the Fuseki cell, then you can query immediately (e.g. re-run the execution cell on a fresh `output.json`).